# 3D Analysis of an Extended Source

In this example we fit the spectum and spatial extension of RX J1713.7-3946 first jointly and then stacked

## Joint Fit

Load all the relevant modules and set the energy unit to `u.TeV` as we are only dealing with H.E.S.S. data here

In [ ]:
import astropy.units as u
import numpy as np
from astromodels.core.model import Model
from astromodels.core.units import get_units
from astromodels.functions import (
    Log_uniform_prior,
    Powerlaw,
    Uniform_prior,
)
from astromodels.sources.extended_source import ExtendedSource
from astropy.coordinates import SkyCoord
from gammapy.data import DataStore
from gammapy.datasets import MapDataset,Datasets
from gammapy.makers import (
    FoVBackgroundMaker,
    MapDatasetMaker,
    SafeMaskMaker,
)
from gammapy.maps import MapAxis, WcsGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    PowerLawSpectralModel,
    SkyModel,
    GaussianSpatialModel,
    FoVBackgroundModel,
)
from regions import CircleSkyRegion
from threeML import DataList,load_analysis_results
from threeML.bayesian.bayesian_analysis import BayesianAnalysis


from gammapy_plugin.converter import AstromodelConverter
from gammapy_plugin.GammapyLike import GammapyLike
from gammapy_plugin.utils.astromodels_functions import Gaussian_on_sphere # TODO remove when correctly implemented in astromodels

get_units().energy = u.TeV


Select all the observations within a `5 deg` radius around RX J1713.7-3946

Prepare the geometry by setting the energy axis and `WcsGeom`

In [ ]:
datastore = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1/")
target_position = SkyCoord.from_name("RX J1713.7-3946").galactic

selection = dict(
    type="sky_circle",
    frame="galactic",
    lon=target_position.l,
    lat=target_position.b,
    radius="5deg",
)
select_obs_tab = datastore.obs_table.select_observations(selection)

obs = datastore.get_observations(select_obs_tab["OBS_ID"])

# Prepare the geometry
energy_axis = MapAxis.from_energy_bounds(0.3, 10.0, 15, unit="TeV")
energy_axis_true = MapAxis.from_energy_bounds(
    0.1, 20, 10, per_decade=True, unit="TeV", name="energy_true"
)
geom = WcsGeom.create(
    skydir=target_position,
    binsz=0.02,
    width=(6 * u.deg, 6 * u.deg),
    frame="galactic",
    axes=[energy_axis],
)



Create the relevant `Makers`

We will also run the `FoVBackgroundMaker` when jointly fitting - this is not needed

We exclude a `1 deg` circle at the source poition for fitting the background models to reduce overestimation of the background

In [ ]:
circle = CircleSkyRegion(center=target_position, radius=1 * u.deg)
regions = [circle]
exclusion_mask = ~geom.region_mask(regions=regions)
maker = MapDatasetMaker(
    selection=["counts", "background", "psf", "edisp", "exposure"],
)
safe_mask_maker = SafeMaskMaker(
    methods=["offset-max", "aeff-max", "bkg-peak"], offset_max="2.3 deg"
)
fov_bkg_maker = FoVBackgroundMaker(method="fit", exclusion_mask=exclusion_mask)

In [ ]:
datasets = Datasets()
gls = []
for o in obs:
    dataset = MapDataset.create(
        geom=geom, energy_axis_true=energy_axis_true, name=f"HESS_{o.obs_id}"
    )
    dataset = maker.run(dataset, o)
    dataset = safe_mask_maker.run(dataset, o)
    bkg_model = FoVBackgroundModel(name = f"{o.obs_id}_bkg",dataset_name= dataset.name)
    dataset.models = [bkg_model]
    dataset = fov_bkg_maker.run(dataset)
    print(f"Bkg norm for HESS_{o.obs_id}: {round(bkg_model.parameters['norm'].value,3)} +/- {round(bkg_model.parameters['norm'].error,3)}")
    datasets.append(dataset)
    gl = GammapyLike(dataset.name, frame="galactic")
    gl.set_datasets(dataset)
    gl.set_background_models(bkg_model)
    gls.append(gl)


In [ ]:
pl = Powerlaw()
spat = Gaussian_on_sphere(
    lon0=target_position.transform_to("galactic").l.deg,
    lat0=target_position.transform_to("galactic").b.deg,
    sigma=0.3,
)
es = ExtendedSource(source_name="rxj1713", spectral_shape=pl, spatial_shape=spat)
pl.index.value = -2
pl.index.prior = Uniform_prior(lower_bound=-3, upper_bound=-1)
pl.K = 6*1e-16 # we use the double differential flux here!
pl.K.prior = Log_uniform_prior(lower_bound=1e-17, upper_bound=1e-14) # fairly tight prior as the fit is really slow
pl.piv.value = 1
pl.piv.free = False
spat.lon0.free = False
spat.lat0.free = False
spat.sigma.free = True
spat.sigma.prior = Uniform_prior(lower_bound=0.1, upper_bound=1.0)
model = Model(es)

conv = AstromodelConverter(model, frame="galactic")



In [ ]:
for gl in gls:
    gl.set_sources("rxj1713")
    gl.set_model(model, conv)

This next cell might hours or days to complete using only a single core. Just skip it and load the result provided in the `result.fits` in the next cell.

```python
ba =BayesianAnalysis(model,DataList(*gls))
ba.set_sampler("ultranest")
ba.sampler.setup()

res = ba.sample(quiet=False)
```

In [ ]:
res = load_analysis_results("data/result.fits")
res.display()

Lets take a look at the differential flux at 1 TeV: 

In [ ]:
flux_1tev = (
    res.optimized_model.extended_sources["rxj1713"].spectrum.main.Powerlaw.K.value
    *res.optimized_model.extended_sources["rxj1713"].spatial_shape.get_total_spatial_integral(z = np.zeros(1))
    *res.optimized_model.extended_sources["rxj1713"].spectrum.main.Powerlaw.K.unit)
hpd_1tev = (np.array(
    res.get_highest_density_posterior_interval(res.optimized_model.extended_sources["rxj1713"].spectrum.main.Powerlaw.K,cl=0.95))
            *res.optimized_model.extended_sources["rxj1713"].spatial_shape.get_total_spatial_integral(z = np.zeros(1))
            *res.optimized_model.extended_sources["rxj1713"].spectrum.main.Powerlaw.K.unit)
flux_1tev,hpd_1tev

which is fairly close to 2.30e-11 +/- 1.00e-12 cm-2 s-1 TeV-1.

Taking a look at the energy integrate counts map of these 15 observations, a 2D-Gaussian might not be the best description.
Additionally only a simple Powerlaw model was fitted so our result is not that bad :)

In [ ]:
datasets.stack_reduce().counts.sum_over_axes(keepdims=False).smooth(0.04*u.deg).plot()

## Stacking the Dataset

We can also speed things up quiet drastically by
1. stacking the datasets
2. only fitting the FoVBackgroundModels before hand and fixing their values


In [ ]:
datasets_stacked = Datasets()

for o in obs:
    dataset = MapDataset.create(
        geom=geom, energy_axis_true=energy_axis_true, name=f"HESS_{o.obs_id}"
    )
    dataset = maker.run(dataset, o)
    dataset = safe_mask_maker.run(dataset, o)
    bkg_model = FoVBackgroundModel(name = f"{o.obs_id}_bkg",dataset_name= dataset.name)
    dataset.models = [bkg_model]
    dataset = fov_bkg_maker.run(dataset)
    datasets_stacked.append(dataset)


Just stack them before passing them to the `GammapyLike` instance or set the `mode="stacked"`

In [ ]:
gl_stacked = GammapyLike("stacked", frame="galactic")
gl_stacked.set_datasets(datasets_stacked,mode = "stacked")


Just the same model as before 

In [ ]:
pl_stacked = Powerlaw()
spat_stacked = Gaussian_on_sphere(
    lon0=target_position.transform_to("galactic").l.deg,
    lat0=target_position.transform_to("galactic").b.deg,
    sigma=0.3,
)
es_stacked = ExtendedSource(source_name="rxj1713_stacked", spectral_shape=pl_stacked, spatial_shape=spat_stacked)
pl_stacked.index.value = -2
pl_stacked.K = 6*1e-16 # we use the double differential flux here!
pl_stacked.piv.value = 1
pl_stacked.piv.free = False
spat_stacked.lon0.free = False
spat_stacked.lat0.free = False
spat_stacked.sigma.free = True
model_stacked = Model(es_stacked)

conv_stacked = AstromodelConverter(model_stacked, frame="galactic")

In [ ]:
gl_stacked.set_model(model_stacked,conv_stacked)

For simplicity just use minuit to minimize the likelihood

In [ ]:
jl = JointLikelihood(model_stacked,DataList(gl_stacked))

In [ ]:
import time

In [ ]:
start = time.time()
res_stacked = jl.fit()
stop = time.time()
print(stop-start)

Compared to the $\sim$ 2 hours `ultranest` took to sample the full posterior using 10 threads this is way way faster

### HOWEVER
This method does not determine the background normalizations during the fit! They are kept fixed to the value determined when running the `FoVBackgroundMaker` $\rightarrow$ if the exclusion region is set too small you will overestimate the background